Set-up bridge between Google Colab and Google Drive.

In [ ]:
# --- Mount Google Drive cleanly ---
from google.colab import drive

# Unmount if already mounted
try:
    drive.flush_and_unmount()
except Exception:
    pass

# Mount
drive.mount("/content/drive", force_remount=True)

# Free space check on Drive
import shutil, os, pathlib
stat = shutil.disk_usage("/content/drive/MyDrive")
print(f"Free space on Drive: {stat.free / (1024**3):.2f} GB")


Clone StarGAN repo to access AFHQ images.

In [ ]:
# --- Fresh clone of repo ---
%cd /content
!rm -rf stargan-v2
!git clone https://github.com/clovaai/stargan-v2.git
%cd /content/stargan-v2

# Make download.sh safe to execute (handle CRLF + perms)
!sed -i -e 's/\r$//' download.sh
!chmod +x download.sh

# Runtime-friendly deps (headless OpenCV avoids GUI conflicts)
!pip -q install munch pyyaml ffmpeg-python scikit-image opencv-python-headless pillow tqdm scipy matplotlib


Set-up directories in both Google drive and Colab.

In [ ]:
# Create MyDrive and Colab Directories
from pathlib import Path
import os, shutil, tarfile, zipfile, random, hashlib, io, math, sys
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)
random.seed(42); np.random.seed(42)

# Central dirs
DRIVE_ROOT   = Path("/content/drive/MyDrive/deepfake_project")
PROGRAMS_DIR = DRIVE_ROOT / "programs"
IMAGES_DIR   = DRIVE_ROOT / "images"        # real + fake live here
OUTPUT_DIR   = DRIVE_ROOT / "output"        # csv reports, logs, plots
ZIPS_DIR     = DRIVE_ROOT / "zips"          # zipped exports

WORK_DIR     = Path("/content/stargan_v2_work")   # local scratch in Colab
STARGAN_DIR  = Path("/content/stargan-v2")        # repo checkout

# Techniques (if/when you balance by family later)
TECHNIQUES = ["stargan","fake_copy_move","fake_inpaint","fake_postproc","fake_splicing","fake_swap_like"]

# Ensure structure on Drive
for d in [PROGRAMS_DIR, IMAGES_DIR, OUTPUT_DIR, ZIPS_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for sub in [
    "real_all_raw", "real_all_dedup", "fake_all_raw",
    "dataset/train/real", "dataset/train/fake",
    "dataset/test/real",  "dataset/test/fake",
    "samples"
]:
    (IMAGES_DIR / sub).mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
print("Folders ready under:", DRIVE_ROOT)


Download AFHQ and AFHQ v2 images and StarGAN model weights into Colab.

In [ ]:
# Download AFHQ Images and Model Weights

!bash download.sh afhq-dataset
!bash download.sh afhq-v2-dataset
!bash download.sh pretrained-network-afhq

# Download Check
!ls -lah data | sed -n '1,120p' || true
!ls -lah expr/checkpoints/afhq | sed -n '1,120p' || true


Create a zip file of all of the images and copy into Google Drive (My Drive).

Display 3 random images each of cat, dog and wild classes.

In [ ]:
# === Show a random sample: 3 images each for cat, dog, wild ===
from pathlib import Path
import random, os
import matplotlib.pyplot as plt
import cv2

DATA_ROOT = Path("/content/stargan-v2/data")
CLASSES   = ["cat", "dog", "wild"]
EXTS      = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Set a seed if you want reproducible samples; comment this out for fresh randomness each run
# random.seed(42)

def find_images_for(species: str):
    # Search any folder named exactly the species anywhere under DATA_ROOT (e.g., train/cat, val/cat, test/cat)
    candidates = []
    for d in DATA_ROOT.rglob(species):
        if d.is_dir():
            for p in d.iterdir():
                if p.is_file() and p.suffix.lower() in EXTS:
                    candidates.append(p)
    return candidates

# Collect 3 random images per class (or fewer if not enough exist)
samples_by_class = {}
for cls in CLASSES:
    imgs = find_images_for(cls)
    if not imgs:
        print(f"[WARN] No images found for class '{cls}' under {DATA_ROOT}")
        samples_by_class[cls] = []
        continue
    k = min(3, len(imgs))
    samples_by_class[cls] = random.sample(imgs, k=k)

# Plot 3 x 3 grid
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 10))
for r, cls in enumerate(CLASSES):
    row_paths = samples_by_class.get(cls, [])
    for c in range(3):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(row_paths):
            p = row_paths[c]
            # Read with OpenCV and convert BGR->RGB
            img = cv2.imread(str(p))
            if img is None:
                ax.set_title(f"{cls}: [unreadable]\n{p.name}", fontsize=9)
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f"{cls} • {p.name}", fontsize=10)
        else:
            ax.set_title(f"{cls}: [no image]", fontsize=9)

plt.tight_layout()
plt.show()


Dedupe AFHQ images based on Name and SHA1. Show duplicate images.

In [ ]:
# Dedupe AFHQ - Rmove duplicate images by Name and SHA1
import os, hashlib, random, shutil
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from datetime import datetime

# ---------------- Config ----------------
random.seed(42)
MAX_SHOW  = 30      # max duplicate pairs to visualize
MAX_PRINT = 50      # max lines to print in console reports
PREFER_SYMLINKS = False
DELETE_DUP_FILES = False   # set True to physically delete duplicate source files

root = Path.cwd()
src_roots = [root/"data/afhq", root/"data/afhq-v2", root/"data/train", root/"data/val"]
src_roots = [p for p in src_roots if p.exists()]
print("[DISCOVERY] Using roots:", [str(p) for p in src_roots])

dst_root  = root/"data/afhq_all"
species   = ["cat","dog","wild"]
splits    = ["train","test"]
train_ratio = 0.70
ratios    = {"train": train_ratio, "test": 1-train_ratio}

# Where to place the zipped copies
ZIPS_DIR  = Path("/content/drive/MyDrive/deepfake_project/zips")
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

# --------------- Helpers ---------------
def sha1(fp: Path, chunk=1<<20):
    h = hashlib.sha1()
    with open(fp, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): return
    if PREFER_SYMLINKS:
        try:
            os.symlink(src, dst); return
        except OSError:
            pass
    shutil.copy2(src, dst)

def show_pairs(pairs, title="Duplicates", max_show=MAX_SHOW):
    if not pairs:
        print(f"[DISPLAY] No {title.lower()} to display.")
        return
    k = min(len(pairs), max_show)
    rows, cols = k, 2
    fig, axes = plt.subplots(rows, cols, figsize=(8, 3*rows))
    if rows == 1:
        kept, dup, note = pairs[0]
        for j, pth in enumerate([kept, dup]):
            ax = axes[j]
            try:
                im = Image.open(pth).convert("RGB")
                ax.imshow(im)
                ax.set_title(("KEPT","DUP")[j] + f": {Path(pth).name}\n{note}", fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error:\n{pth}\n{e}", ha='center', va='center')
            ax.axis('off')
        plt.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()
        return
    for i in range(k):
        kept, dup, note = pairs[i]
        for j, pth in enumerate([kept, dup]):
            ax = axes[i, j]
            try:
                im = Image.open(pth).convert("RGB")
                ax.imshow(im)
                role = "KEPT" if j == 0 else "DUP"
                ax.set_title(f"{role}: {Path(pth).name}\n{note}", fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error:\n{pth}\n{e}", ha='center', va='center')
            ax.axis('off')
    plt.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()

def zip_and_move(folder: Path, label: str, out_dir: Path) -> Path:
    if not folder.exists() or not any(folder.rglob("*")):
        raise RuntimeError(f"Nothing to zip in {folder}")
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base_name = Path(f"/content/{label}-{stamp}")   # temp path (no extension)
    archive_path = shutil.make_archive(
        base_name=str(base_name),
        format="zip",
        root_dir=folder.parent,
        base_dir=folder.name
    )
    out_dir.mkdir(parents=True, exist_ok=True)
    final = out_dir / Path(archive_path).name
    shutil.move(archive_path, final)
    print(f"[ZIP] Wrote: {final} ({final.stat().st_size / (1024**3):.2f} GB)")
    return final

# 1) Collect
all_paths = []              # list of tuples (species, Path)
pre_counts = {s: 0 for s in species}
for s in species:
    for src in src_roots:
        for split_guess in ["train","test","val",""]:
            d = src/split_guess/s if split_guess else src/s
            if d.exists():
                found = [(s, p) for p in d.rglob("*") if p.suffix.lower() in EXTS and p.is_file()]
                all_paths += found
                pre_counts[s] += len(found)
print(f"[DISCOVERY] Collected {len(all_paths):,} files across roots.")
print("[COUNTS: pre-dedupe by species]", pre_counts, "| total:", sum(pre_counts.values()))

# 2) Dedup pass A: by filename (within species)
seen_names = {s: {} for s in species}
name_dups, kept_after_name = [], {s: [] for s in species}
for s, p in all_paths:
    nm = p.name
    if nm in seen_names[s]:
        name_dups.append({"species": s, "dup": p, "kept": seen_names[s][nm]})
    else:
        seen_names[s][nm] = p
        kept_after_name[s].append(p)

# 3) Dedup pass B: by SHA-1 (within species)
hash_maps = {s: {} for s in species}
sha_dups  = []
for s in species:
    for p in kept_after_name[s]:
        try:
            h = sha1(p)
        except Exception:
            continue
        if h in hash_maps[s]:
            sha_dups.append({"species": s, "dup": p, "kept": hash_maps[s][h], "sha1": h})
        else:
            hash_maps[s][h] = p

# Optionally remove duplicate files at the source
if DELETE_DUP_FILES:
    to_delete = [rec["dup"] for rec in name_dups] + [rec["dup"] for rec in sha_dups]
    deleted = 0
    for fp in to_delete:
        try:
            fp.unlink()
            deleted += 1
        except Exception:
            pass
    print(f"[DELETE] Removed {deleted} duplicate files from source locations.")

# Final kept set (unique by SHA-1 within species)
final_kept = {s: sorted(set(hash_maps[s].values())) for s in species}

# Reports + CSVs
pd.DataFrame(name_dups).to_csv(root/"name_dups.csv", index=False)
pd.DataFrame(sha_dups ).to_csv(root/"sha_dups.csv",  index=False)

print("\n=== Duplicate Removal Report ===")
print(f"Same filename removed: {len(name_dups):,}")
for rec in name_dups[:MAX_PRINT]:
    print(f"[NAME DUP] species={rec['species']}  dup={rec['dup'].name}  -> kept={rec['kept'].name}")
if len(name_dups) > MAX_PRINT:
    print(f"... ({len(name_dups)-MAX_PRINT} more)")

print(f"\nSame SHA-1 removed: {len(sha_dups):,}")
for rec in sha_dups[:MAX_PRINT]:
    print(f"[SHA1 DUP] species={rec['species']}  dup={rec['dup'].name}  -> kept={rec['kept'].name}  sha1={rec['sha1'][:12]}...")
if len(sha_dups) > MAX_PRINT:
    print(f"... ({len(sha_dups)-MAX_PRINT} more)")

# Visualization of duplicates
name_pairs = [(rec["kept"], rec["dup"], f"species={rec['species']} (same filename)") for rec in name_dups]
sha_pairs  = [(rec["kept"], rec["dup"], f"species={rec['species']} (same SHA1)") for rec in sha_dups]
show_pairs(name_pairs, title="Name Duplicates (kept vs dup)")
show_pairs(sha_pairs,  title="SHA-1 Duplicates (kept vs dup)")

# 4) Post-dedupe counts
post_counts = {s: len(final_kept[s]) for s in species}
print("\n[COUNTS: post-dedupe by species]", post_counts, "| total:", sum(post_counts.values()))

pd.DataFrame(
    [{"phase":"pre", **pre_counts, "total": sum(pre_counts.values())},
     {"phase":"post", **post_counts, "total": sum(post_counts.values())}]
).to_csv(root/"afhq_all_counts_pre_post.csv", index=False)

# 5) Random split (70/30) + write unique set (hash-prefixed filenames)
#    - Randomized per species to honor requested split ratio globally
for split in splits:
    for s in species:
        (dst_root/split/s).mkdir(parents=True, exist_ok=True)

assignments = []  # rows for manifest
for s in species:
    items = list(final_kept[s])
    random.shuffle(items)
    k_train = int(round(train_ratio * len(items)))
    train_set = items[:k_train]
    test_set  = items[k_train:]
    for split, group in [("train", train_set), ("test", test_set)]:
        for p in group:
            try:
                h = sha1(p)
            except Exception:
                h = None
            out_name = (f"{h[:8]}__{p.name}") if h else p.name
            dst = dst_root/split/s/out_name
            link_or_copy(p, dst)
            assignments.append({"split": split, "species": s, "src": str(p), "dst": str(dst), "sha1": h})

# 6) Final counts + manifest
rows = []
result_counts = {split: {s: 0 for s in species} for split in splits}
for split in splits:
    for s in species:
        files = [x for x in (dst_root/split/s).glob('*') if x.is_file()]
        result_counts[split][s] = len(files)
        for f in files:
            rows.append({"split": split, "species": s, "dst": str(f)})

print("\n[DATASET COUNTS]")
for split in splits:
    print(split, result_counts[split], "| total:", sum(result_counts[split].values()))

manifest_df = pd.DataFrame(assignments)
manifest_path = root/"afhq_all_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print("[WRITE] Manifest:", manifest_path)

# 7) Zip train/ and test/ and move to Drive zips folder
train_zip = zip_and_move(dst_root/"train", label="afhq_all-train", out_dir=ZIPS_DIR)
test_zip  = zip_and_move(dst_root/"test",  label="afhq_all-test",  out_dir=ZIPS_DIR)
print("[DONE] Zipped archives moved to:", ZIPS_DIR)


Create two random train sets and 1 random test set of images to apply StarGAN for GAN deepfakes. This process is time-consuming and requires heavy processing.

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# Make TWO separate StarGAN v2 runs for TRAIN (distinct refs/output), plus one for TEST
from pathlib import Path
import os, random, shutil

base = Path.cwd()
real_root = base/"data/afhq_all"

def make_ref_tree(split, tag, n_per_domain=10, seed=None):
    """
    Build a reference tree at tmp_refs/{split}_{tag}/{cat,dog,wild}
    Using up to n_per_domain random images per domain.
    """
    if seed is not None:
        random.seed(seed)

    ref_root = base/f"tmp_refs/{split}_{tag}"
    for dom in ["cat","dog","wild"]:
        d = ref_root/dom
        d.mkdir(parents=True, exist_ok=True)
        # clear previous refs for reproducibility
        for f in d.glob("*"): f.unlink()

        imgs = [p for p in (real_root/split/dom).glob("*")
                if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:min(n_per_domain, len(imgs))]:
            try:
                os.symlink(p, d/p.name)
            except OSError:
                shutil.copy2(p, d/p.name)
    return ref_root

# ===== Build separate reference sets =====
# You can tweak seeds to ensure different grids
print("Created refs at:", make_ref_tree("train", "run1", n_per_domain=10, seed=123))
print("Created refs at:", make_ref_tree("train", "run2", n_per_domain=10, seed=456))
print("Created refs at:", make_ref_tree("test",  "main", n_per_domain=10, seed=789))

# ===== Run StarGAN v2 sampling =====
# (stay in the repo root)
print("Changing dir to /content/stargan-v2 ...")
# %cd /content/stargan-v2

# TRAIN - Run 1
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir    expr/results/afhq_all_fake_stargan/train_run1 \
  --src_dir       data/afhq_all/train \
  --ref_dir       tmp_refs/train_run1

# TRAIN - Run 2
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir    expr/results/afhq_all_fake_stargan/train_run2 \
  --src_dir       data/afhq_all/train \
  --ref_dir       tmp_refs/train_run2

# TEST - single run (unchanged)
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir    expr/results/afhq_all_fake_stargan/test_all \
  --src_dir       data/afhq_all/test \
  --ref_dir       tmp_refs/test_main


The output of the StarGAN is a tile image which has to be cut or parsed to get the individual deepfake images. There are two tile images for train and 1 tile image for test.

In [ ]:
# === Zip all StarGAN tile images and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, os

ZIPS_DIR = Path("/content/drive/MyDrive/deepfake_project/zips")
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

# Stage only images to keep archive small
STAGE_DIR = Path("/content/_tiles_stage")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def is_img(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

copied = 0
for p in RESULT_ROOT.rglob("*"):
    if is_img(p):
        rel = p.relative_to(RESULT_ROOT)     # keep subfolder structure
        dst = STAGE_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)
        copied += 1

if copied == 0:
    raise RuntimeError(f"No image tiles found under {RESULT_ROOT}")

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/afhq_all_fake_stargan_tiles-{stamp}")  # no extension here
zip_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(STAGE_DIR.parent),
    base_dir=STAGE_DIR.name
)

final_path = ZIPS_DIR / Path(zip_path).name
shutil.move(zip_path, final_path)

# Optional: clean up staging
shutil.rmtree(STAGE_DIR, ignore_errors=True)

size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Zipped {copied:,} tile images to: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


This is to show an example of what tile looks like. The first row and column are real images with blended image occurring in the middle.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

RESULT_ROOT = Path("/content/stargan-v2/expr/results/afhq_all_fake_stargan")
run_dir = RESULT_ROOT / "train_run1"
exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

# Find the first image under train_run1 (recursively)
img_path = None
for p in run_dir.rglob("*"):
    if p.is_file() and p.suffix.lower() in exts:
        img_path = p
        break

if not img_path:
    raise FileNotFoundError(f"No images found under {run_dir}")

print("Showing:", img_path)
img = Image.open(img_path).convert("RGB")
plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()
